# PulseInsightAI - Exploratory Data Analysis(EDA)
## Objective: 
Explore The CDC BRFSS dataset to :
- Understand the Structure and Shape of the Data.
- Identify Missing Values.
- Analyze Target Variable Distribution.
- Discover Key patterns and correleation.
- Check for class imbalance.


In [2]:
import pandas as pd
import numpy as np

print("Loading dataset... this will take 1-2 minutes")

df = pd.read_sas("../data/raw/LLCP2024.XPT", 
                  format="xport", 
                  encoding="utf-8")

print(f"Dataset loaded!")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
df.head(3)

Loading dataset... this will take 1-2 minutes
Dataset loaded!
Rows: 457670
Columns: 301


,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO,_PSU,CTELENM1,...,_LCSCTSN,_LCSPSTF,DRNKANY6,DROCDY4_,_RFBING6,_DRNKWK3,_RFDRHV9,_FLSHOT7,_PNEUMO3,_AIDTST4
0,1.0,2.0,02282024,02,28,2024,1100.0,2024000001,2.024000e+09,1.0,...,NaN,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,2.0,2.0
1,1.0,2.0,02212024,02,21,2024,1100.0,2024000002,2.024000e+09,1.0,...,4.0,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0
2,1.0,2.0,02212024,02,21,2024,1100.0,2024000003,2.024000e+09,1.0,...,4.0,2.0,1.0,1.000000e+02,2.0,1.400000e+03,1.0,NaN,NaN,2.0


In [3]:
print("Showing Every Columns of the Dataset:")
'''
for i, col in enumerate(df.columns):
    print(f"{i} - {col}")
'''
print(df.columns.to_list())


Showing Every Columns of the Dataset:
['_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO', '_PSU', 'CTELENM1', 'PVTRESD1', 'COLGHOUS', 'STATERE1', 'CELPHON1', 'LADULT1', 'NUMADULT', 'RESPSLC1', 'LANDSEX3', 'SAFETIME', 'CTELNUM1', 'CELLFON5', 'CADULT1', 'CELLSEX3', 'PVTRESD3', 'CCLGHOUS', 'CSTATE1', 'LANDLINE', 'HHADULT', 'SEXVAR', 'GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'PRIMINS2', 'PERSDOC3', 'MEDCOST1', 'CHECKUP1', 'EXERANY2', 'LASTDEN4', 'RMVTETH4', 'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'ASTHMA3', 'ASTHNOW', 'CHCSCNC1', 'CHCOCNC1', 'CHCCOPD3', 'ADDEPEV3', 'CHCKDNY2', 'HAVARTH4', 'DIABETE4', 'DIABAGE4', 'MARITAL', 'EDUCA', 'RENTHOM1', 'NUMHHOL4', 'NUMPHON4', 'CPDEMO1C', 'VETERAN3', 'EMPLOY1', 'CHILDREN', 'INCOME3', 'PREGNANT', 'WEIGHT2', 'HEIGHT3', 'DEAF', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES', 'DIFFALON', 'HADMAM', 'HOWLONG', 'CERVSCRN', 'CRVCLCNC', 'CRVCLPAP', 'CRVCLHPV', 'HADHYST2', 'HADSIGM4', 'COLNSIGM', 'COLNTES1', 'SIGMTES1', 'LASTSIG4', 'COLN

In [4]:
print("Showing Count of datatype of the features:")
print(df.dtypes.value_counts())
print("Showing all the Object features:")
print(df.select_dtypes(include='object').columns.to_list())

Showing Count of datatype of the features:
float64    296
object       5
Name: count, dtype: int64
Showing all the Object features:
['IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'SEQNO']


## Key Finding #1
- 301 total columns
- 296 numeric (float64) — 98% already numeric
- 5 text (object) — need encoding later
- Minimal preprocessing required for data types

In [5]:
# Missing values count
print(f"Total columns with missing values: {df.isnull().any().sum()}")
print(f"Total complete columns: {df.notnull().all().sum()}")
missing = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_percent
})

missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print(missing_df.head(20))

Total columns with missing values: 249
Total complete columns: 52
          Missing Count   Missing %
RCSXBRTH         457670  100.000000
RCSGEND1         457670  100.000000
COLGHOUS         457656   99.996941
CSRVCTL2         457088   99.872834
ICFQSTVR         456931   99.838530
CCLGHOUS         456244   99.688422
HPVDSHT          455650   99.558634
CSRVINST         455045   99.426443
NOBCUSE8         454899   99.394542
LASTSIG4         454443   99.294907
CSRVCLIN         454128   99.226080
CSRVDEIN         454126   99.225643
CSRVINSR         454123   99.224987
CSRVRTRN         454118   99.223895
CSRVSUM          454114   99.223021
CSRVDOC1         454108   99.221710
HPVADSH1         454043   99.207508
CASTHNO2         452865   98.950117
NUMPHON4         452048   98.771604
CSRVPAIN         451664   98.687701


## Key Finding #2 — Missing Values
- 249 out of 301 columns have missing values (83%)
- Only 52 columns are complete
- 2 columns are 100% empty → will be dropped
- Most high-missing columns are optional survey questions
- Strategy: Drop columns with more than 50% missing in preprocessing